# 4.2 — KNN Real World Problem: Credit Risk Prediction
## Chennai Bank — Loan Default Prediction

**Problem:** A bank in Chennai wants to predict whether a loan applicant will default.  
**Why it matters:**
- If we **miss a defaulter** (False Negative) → bank loses money on bad loans
- If we **flag a good customer** (False Positive) → customer is wrongly rejected

**Our job:** Build a KNN model that catches as many defaulters as possible, without being too aggressive in rejecting good customers.

---
**Dataset features:**
- `income` — monthly income in ₹
- `age` — age of applicant
- `num_loans` — number of existing loans
- `credit_score` — credit score (300–850, higher = better)
- `default` — target: 1 = defaulted, 0 = repaid

## Step 0 — Import Libraries

In [ ]:
# numpy — fast numerical operations (arrays, random numbers, math)
import numpy as np

# pandas — load and manipulate tabular data (DataFrames)
import pandas as pd

# matplotlib — plotting graphs
import matplotlib.pyplot as plt

# seaborn — nicer statistical plots built on top of matplotlib
import seaborn as sns

# KNeighborsClassifier — the KNN algorithm from sklearn
from sklearn.neighbors import KNeighborsClassifier

# StandardScaler — scales features to mean=0, std=1
# MANDATORY for KNN because it uses distance
from sklearn.preprocessing import StandardScaler

# train_test_split — splits data into train and test sets
# cross_val_score — runs k-fold cross validation
# StratifiedKFold — k-fold that preserves class ratio in each fold
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold

# Evaluation metrics
# classification_report — precision, recall, f1 for all classes at once
# confusion_matrix — TP, TN, FP, FN table
# roc_auc_score — AUC score (threshold-independent performance)
# precision_score, recall_score, f1_score — individual metric functions
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_score, recall_score, f1_score
)

# Pipeline — chains multiple steps (scaler → model) into one object
# Prevents data leakage — scaler learns only from train data
from sklearn.pipeline import Pipeline

# Set random seed so results are reproducible every time you run
np.random.seed(42)

print("All libraries loaded successfully.")

## Step 1 — Create the Dataset

In [ ]:
# Set seed again here for reproducibility of this specific cell
np.random.seed(42)

# Number of loan applicants we're simulating
n = 500

# --- Generate feature columns ---

# Income: normally distributed around ₹45,000/month
# std=15000 means most people earn between ₹30k and ₹60k
# .clip(10000, 150000) ensures no impossible values (no negative income)
income = np.random.normal(45000, 15000, n).clip(10000, 150000)

# Age: random integer between 22 and 64 (working age adults)
age = np.random.randint(22, 65, n)

# Number of existing loans: 0 to 4
# More existing loans = higher risk of default
loans = np.random.randint(0, 5, n)

# Credit score: 300 (poor) to 850 (excellent)
# Higher = better credit history = less likely to default
score = np.random.randint(300, 850, n)

# --- Generate target (default: 1=defaulted, 0=repaid) ---
# Rule: default if income is low OR too many loans OR poor credit
# This mimics how real defaults happen
default = ((income < 30000) | (loans > 3) | (score < 500)).astype(int)

# Add 10% random noise — real data is never perfectly clean
# This flips 10% of labels randomly (some good customers default, some bad ones don't)
noise_mask = np.random.rand(n) < 0.10  # True for 10% of rows
default = np.where(noise_mask, 1 - default, default)  # flip those rows

# --- Build the DataFrame ---
credit_df = pd.DataFrame({
    'income': income,
    'age': age,
    'num_loans': loans,
    'credit_score': score,
    'default': default
})

print("Dataset shape:", credit_df.shape)  # should be (500, 5)
print(f"Default rate: {credit_df['default'].mean()*100:.1f}%")  # how many defaulted
print("\nFirst 5 rows:")
credit_df.head()

## Step 2 — Explore the Data (EDA)

In [ ]:
# Always inspect a new dataset before modelling

print("=== Basic Info ===")
print(credit_df.info())         # column types, non-null counts

print("\n=== Statistics ===")
print(credit_df.describe().round(2))  # mean, std, min, max per column

print("\n=== Missing Values ===")
print(credit_df.isnull().sum())  # count NaN per column — should be 0 here

print("\n=== Target Distribution ===")
print(credit_df['default'].value_counts())
# 0 = repaid, 1 = defaulted
# If heavily imbalanced (e.g. 95% vs 5%), we'd need special handling

In [ ]:
# Visualise feature distributions — split by default vs no default
# This tells us which features separate the two classes

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
features = ['income', 'age', 'num_loans', 'credit_score']

for ax, feat in zip(axes.flat, features):
    # Plot defaulters (class=1) in red
    credit_df[credit_df['default'] == 1][feat].hist(
        ax=ax, alpha=0.6, color='coral', label='Defaulted', bins=20
    )
    # Plot non-defaulters (class=0) in blue
    credit_df[credit_df['default'] == 0][feat].hist(
        ax=ax, alpha=0.6, color='steelblue', label='Repaid', bins=20
    )
    ax.set_title(f'{feat} distribution by outcome', fontsize=11)
    ax.set_xlabel(feat)
    ax.legend()

plt.suptitle("Feature Distributions — Defaulted vs Repaid", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

# What to look for:
# If red and blue histograms overlap a lot → feature is weak
# If they separate well → feature is strong for prediction

In [ ]:
# Check correlation of each feature with default
# This gives a quick ranking of feature usefulness

correlations = credit_df.corr()['default'].drop('default').sort_values()

print("Correlation with default outcome:")
print(correlations.round(3))

# Interpret:
# Positive correlation = higher value → more likely to default
# Negative correlation = higher value → less likely to default

plt.figure(figsize=(7, 4))
correlations.plot(
    kind='barh',
    color=['coral' if c > 0 else 'steelblue' for c in correlations]
)
plt.axvline(0, color='black', linewidth=0.8)
plt.title("Feature Correlation with Default")
plt.xlabel("Pearson Correlation")
plt.tight_layout()
plt.show()

## Step 3 — Split into X and y

In [ ]:
# X = features (everything the model uses to predict)
# We drop 'default' because that's what we're trying to predict
X_credit = credit_df.drop(columns=['default'])

# y = target (what we're predicting: 0=repaid, 1=defaulted)
y_credit = credit_df['default']

print("X shape:", X_credit.shape)  # (500, 4) — 500 rows, 4 features
print("y shape:", y_credit.shape)  # (500,)   — 500 labels
print("\nFeatures:", list(X_credit.columns))
print("Target:  ", y_credit.name)

## Step 4 — Train/Test Split

In [ ]:
# Split the data into training (80%) and testing (20%) sets
# 
# test_size=0.2 → 20% goes to test (100 rows), 80% for training (400 rows)
# random_state=42 → same split every time you run (reproducible)
# stratify=y_credit → ensures both train and test have the same default rate
#   Without stratify: by luck, test might have 40% defaulters, train only 30%
#   With stratify: both sets have the same ratio — fair evaluation

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_credit, y_credit,
    test_size=0.2,
    random_state=42,
    stratify=y_credit
)

print(f"Training set:  {X_train_c.shape[0]} applicants")
print(f"Test set:      {X_test_c.shape[0]} applicants")

# Verify stratification worked — default rates should be similar
print(f"\nDefault rate — Full:  {y_credit.mean()*100:.1f}%")
print(f"Default rate — Train: {y_train_c.mean()*100:.1f}%")
print(f"Default rate — Test:  {y_test_c.mean()*100:.1f}%")
# All three should be close to each other

## Step 5 — Build Pipeline (Scaler + KNN)

In [ ]:
# WHY PIPELINE?
# Bad way: scale entire dataset → split → train
#   Problem: scaler has seen test data → data leakage → fake good results
# Good way: Pipeline fits scaler ONLY on train data, applies to test
#   This is what Pipeline does automatically

# Build the pipeline with two steps:
# Step 1: StandardScaler — converts every feature to mean=0, std=1
#   income (₹10k-₹150k) and credit_score (300-850) are on very different scales
#   Without scaling, income dominates distance → credit_score barely matters
# Step 2: KNeighborsClassifier — the actual KNN model
#   n_neighbors=5 → look at 5 nearest applicants and take majority vote
#   metric='euclidean' → straight-line distance (default, works well generally)

credit_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5, metric='euclidean'))
])

# .fit() → scaler learns mean/std from X_train_c only
#         → KNN stores all training points
credit_pipeline.fit(X_train_c, y_train_c)

# .predict() → scales X_test_c using train stats → finds 5 nearest neighbors
y_pred_c = credit_pipeline.predict(X_test_c)

# .predict_proba()[:,1] → probability of being a defaulter (class=1)
# We use this for AUC and threshold tuning
y_prob_c = credit_pipeline.predict_proba(X_test_c)[:, 1]

print("Model trained and predictions made.")
print(f"First 10 predictions:     {y_pred_c[:10]}")
print(f"First 10 probabilities:   {y_prob_c[:10].round(2)}")
# Notice: prediction = 1 when probability >= 0.5

## Step 6 — Evaluate the Model

In [ ]:
# Classification report — gives all metrics at once
# precision = of all predicted defaulters, how many actually defaulted?
# recall    = of all actual defaulters, how many did we catch?
# f1-score  = harmonic mean of precision and recall (balanced metric)
# support   = number of actual samples in each class

print("=== Classification Report (K=5) ===")
print(classification_report(
    y_test_c,           # actual labels
    y_pred_c,           # predicted labels
    target_names=['Repaid (0)', 'Defaulted (1)']  # human-readable names
))

# AUC score — threshold independent
# Measures: if we pick one random defaulter and one random repayer,
# how often does the model rank the defaulter higher?
# 0.5 = random guessing, 1.0 = perfect
auc = roc_auc_score(y_test_c, y_prob_c)
print(f"ROC-AUC Score: {auc:.4f}")

In [ ]:
# Confusion Matrix — most important diagnostic tool
# Rows = actual class, Columns = predicted class
#
#                 Predicted Repaid  Predicted Default
# Actual Repaid       TN                 FP
# Actual Default      FN                 TP
#
# TN = True Negative  → correctly said 'will repay'
# FP = False Positive → wrongly said 'will default' (false alarm)
# FN = False Negative → missed a defaulter (DANGEROUS for bank)
# TP = True Positive  → correctly caught a defaulter

cm = confusion_matrix(y_test_c, y_pred_c)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,          # show numbers inside cells
    fmt='d',             # integer format
    cmap='Blues',        # color scheme
    xticklabels=['Predicted Repaid', 'Predicted Default'],
    yticklabels=['Actual Repaid', 'Actual Default']
)
plt.title("Confusion Matrix — KNN (K=5)")
plt.tight_layout()
plt.show()

# Extract individual values for plain-English interpretation
tn, fp, fn, tp = cm.ravel()  # .ravel() flattens 2D matrix to 1D

print(f"True Negatives  (repaid, correctly approved): {tn}")
print(f"False Positives (repaid, wrongly rejected):   {fp}  ← lost good customers")
print(f"False Negatives (defaulted, we missed):       {fn}  ← BANK LOSES MONEY")
print(f"True Positives  (defaulted, correctly caught):{tp}")
print(f"\nOut of {fn+tp} actual defaulters, we caught {tp} ({tp/(fn+tp)*100:.1f}%)")

## Step 7 — Find the Best K

In [ ]:
# K=5 was just a starting guess. We need to find the BEST K.
#
# Strategy:
# - Try K from 1 to 20
# - For each K, run 5-fold cross-validation on training data
# - Pick K with highest CV F1 score
# - Why F1? Because we care about both precision AND recall (imbalanced stakes)
# - Why CV? Because test set should only be touched ONCE at the very end

k_range = range(1, 21)  # try K = 1, 2, 3, ..., 20

cv_f1_scores    = []    # cross-validated F1 per K
train_acc_scores = []   # training accuracy per K
test_acc_scores  = []   # test accuracy per K

# StratifiedKFold — 5 folds, shuffled, preserving default rate in each fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for k in k_range:
    # Build a fresh pipeline for this K value
    pipe_k = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k, metric='euclidean'))
    ])
    
    # Train on full training set and score on train (to check overfitting)
    pipe_k.fit(X_train_c, y_train_c)
    train_acc_scores.append(pipe_k.score(X_train_c, y_train_c))  # train accuracy
    test_acc_scores.append(pipe_k.score(X_test_c, y_test_c))     # test accuracy
    
    # Cross-validation F1 — more reliable than single test score
    # scoring='f1' → use F1 score for each fold
    cv_f1 = cross_val_score(pipe_k, X_train_c, y_train_c, cv=cv, scoring='f1')
    cv_f1_scores.append(cv_f1.mean())  # average across 5 folds

# Best K = the one with highest cross-validated F1
best_k = list(k_range)[np.argmax(cv_f1_scores)]
print(f"Best K by cross-validated F1: K = {best_k}")
print(f"CV F1 at best K: {max(cv_f1_scores):.4f}")

In [ ]:
# Plot the K selection curves
plt.figure(figsize=(10, 5))

# Train accuracy — usually decreases as K increases (less overfitting)
plt.plot(k_range, train_acc_scores, label='Train Accuracy', 
         color='steelblue', marker='o', markersize=4)

# Test accuracy — may be noisy because it's one split
plt.plot(k_range, test_acc_scores, label='Test Accuracy', 
         color='coral', marker='o', markersize=4)

# CV F1 — most reliable signal for best K
plt.plot(k_range, cv_f1_scores, label='CV F1 Score (reliable)', 
         color='green', marker='s', markersize=4, linestyle='--')

# Highlight the best K
plt.axvline(best_k, color='black', linestyle=':', linewidth=1.5, 
            label=f'Best K = {best_k}')

plt.title("Finding the Best K — Overfitting (K=1) vs Underfitting (K=large)")
plt.xlabel("K (number of neighbors)")
plt.ylabel("Score")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("What the plot tells us:")
print("  K=1: train accuracy near 100% (memorises all training data = overfitting)")
print("  K=large: both scores drop (too smooth, misses patterns = underfitting)")
print(f"  K={best_k}: sweet spot — balanced bias and variance")

## Step 8 — Retrain with Best K and Final Evaluation

In [ ]:
# Now retrain with the best K we found
# This is the FINAL model — only now do we look at test results for real

best_credit_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=best_k, metric='euclidean'))
])

# Fit on training data
best_credit_pipeline.fit(X_train_c, y_train_c)

# Predict class labels (0 or 1) on test set
y_pred_best = best_credit_pipeline.predict(X_test_c)

# Predict probability of default (class=1) — needed for AUC + threshold tuning
# predict_proba returns [[prob_class0, prob_class1], ...]
# [:,1] takes the second column = probability of being a defaulter
y_prob_best = best_credit_pipeline.predict_proba(X_test_c)[:, 1]

print(f"=== Final Model — KNN with K={best_k} ===")
print(classification_report(
    y_test_c, y_pred_best,
    target_names=['Repaid (0)', 'Defaulted (1)']
))
print(f"ROC-AUC: {roc_auc_score(y_test_c, y_prob_best):.4f}")

## Step 9 — Should the Bank Use Threshold = 0.5 or Lower?

In [ ]:
# Default threshold = 0.5
# If probability of default >= 0.5 → predict 'will default'
#
# But is 0.5 right for a bank?
#
# Think about the costs:
# - False Negative (miss a defaulter) → bank gives loan → doesn't get money back → ₹ LOST
# - False Positive (flag a good customer) → customer rejected → bank loses business
#
# In most banks, missing a defaulter is MORE COSTLY than a false alarm
# So we should LOWER the threshold → flag more people as defaulters → catch more bad loans
# Trade-off: more false positives (good customers rejected) but fewer bad loans funded

# Test thresholds from 0.1 to 0.9
thresholds = np.arange(0.10, 0.90, 0.05)

precisions   = []
recalls      = []
f1s          = []
missed       = []   # False Negatives = defaulters we missed
false_alarms = []   # False Positives = good customers wrongly rejected

for thresh in thresholds:
    # Convert probability to class label using this threshold
    # If prob >= thresh → predict 1 (default), else 0 (repaid)
    y_thresh = (y_prob_best >= thresh).astype(int)
    
    # Calculate metrics at this threshold
    precisions.append(precision_score(y_test_c, y_thresh, zero_division=0))
    recalls.append(recall_score(y_test_c, y_thresh, zero_division=0))
    f1s.append(f1_score(y_test_c, y_thresh, zero_division=0))
    
    # Count business-critical errors
    cm_t = confusion_matrix(y_test_c, y_thresh)
    missed.append(cm_t[1][0])       # FN: bottom-left of confusion matrix
    false_alarms.append(cm_t[0][1]) # FP: top-right of confusion matrix

# Plot precision/recall/F1 vs threshold
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(thresholds, precisions, color='steelblue', label='Precision', marker='o', markersize=3)
ax1.plot(thresholds, recalls,    color='coral',     label='Recall',    marker='o', markersize=3)
ax1.plot(thresholds, f1s,        color='green',     label='F1 Score',  marker='s', markersize=3)
ax1.axvline(0.5, color='gray', linestyle='--', label='Default threshold (0.5)')
ax1.set_title("Precision / Recall / F1 vs Threshold")
ax1.set_xlabel("Threshold")
ax1.set_ylabel("Score")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot business impact — missed defaulters and false alarms
ax2.plot(thresholds, missed,       color='red',    linewidth=2, label='Missed Defaulters (FN) ← costly')
ax2.plot(thresholds, false_alarms, color='orange', linewidth=2, label='False Alarms (FP) ← lost customers')
ax2.axvline(0.5,  color='gray',  linestyle='--', label='Default (0.5)')
ax2.axvline(0.35, color='black', linestyle=':',  label='Conservative (0.35)')
ax2.set_title("Business Impact vs Threshold")
ax2.set_xlabel("Threshold")
ax2.set_ylabel("Count of errors")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Find the index closest to threshold=0.5 and 0.35 for comparison
idx_05 = np.argmin(np.abs(thresholds - 0.50))
idx_035 = np.argmin(np.abs(thresholds - 0.35))

print("=== Threshold Comparison ===")
print(f"\nAt threshold = 0.50 (default):")
print(f"  Defaulters missed (FN):        {missed[idx_05]}")
print(f"  Good customers rejected (FP):  {false_alarms[idx_05]}")
print(f"  Recall:    {recalls[idx_05]:.2f}")
print(f"  Precision: {precisions[idx_05]:.2f}")

print(f"\nAt threshold = 0.35 (conservative):")
print(f"  Defaulters missed (FN):        {missed[idx_035]}")
print(f"  Good customers rejected (FP):  {false_alarms[idx_035]}")
print(f"  Recall:    {recalls[idx_035]:.2f}")
print(f"  Precision: {precisions[idx_035]:.2f}")

In [ ]:
# ANSWER: Should the bank use 0.5 or lower?
#
# ANSWER: YES — the bank should use a LOWER threshold, around 0.35
#
# Reason:
# A bad loan costs the bank the entire loan amount (e.g. ₹5 lakhs)
# A rejected good customer costs the bank the profit margin (e.g. ₹10,000)
# The asymmetry means: it's much better to flag too many than to miss defaulters
#
# At 0.35: we catch more defaulters (higher recall)
#   Trade-off: some good customers are wrongly rejected
#   The bank can implement a second manual review stage for borderline cases
#
# This is the key insight: threshold choice is a BUSINESS decision, not just a math one

print("RECOMMENDATION:")
print("Use threshold = 0.35 for the bank's loan decisions.")
print("This catches more defaulters, at the cost of some false alarms.")
print("Borderline cases (0.35-0.5) can go to manual review by a loan officer.")

## Step 10 — Cross-Validation (Is the Model Stable?)

In [ ]:
# Cross-validation checks if our model is consistently good
# or just got lucky with one particular train/test split
#
# 5-fold CV: split training data into 5 parts
# Round 1: train on parts 2,3,4,5 → test on part 1
# Round 2: train on parts 1,3,4,5 → test on part 2
# ... and so on for 5 rounds
# Report: mean ± std across all 5 rounds
# Small std = consistent model. Large std = unstable.

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"=== 5-Fold Cross-Validation (K={best_k}) ===")
for metric_name in ['accuracy', 'f1', 'roc_auc']:
    scores = cross_val_score(
        best_credit_pipeline,  # our final pipeline
        X_train_c,             # ONLY training data — test is untouched
        y_train_c,
        cv=cv,
        scoring=metric_name
    )
    print(f"{metric_name:12s}: {scores.mean():.4f} ± {scores.std():.4f}"
          f"  | Folds: {[round(s,3) for s in scores]}")

print("\nInterpretation:")
print("  Small std (< 0.03) = model is stable across different subsets of data")
print("  Large std (> 0.05) = model is sensitive to which rows end up in train")

## Step 11 — Predict for a New Loan Applicant

In [ ]:
# Predict for 3 real loan applicants who just walked into the Chennai branch

new_applicants = pd.DataFrame([
    # income,  age, num_loans, credit_score
    {'income': 20000, 'age': 27, 'num_loans': 4, 'credit_score': 420},  # Ravi — risky
    {'income': 75000, 'age': 40, 'num_loans': 1, 'credit_score': 780},  # Priya — safe
    {'income': 35000, 'age': 32, 'num_loans': 2, 'credit_score': 520},  # Meena — borderline
])

# Get probability of default for each applicant
# Pipeline automatically scales using stats from training data
probs = best_credit_pipeline.predict_proba(new_applicants)[:, 1]

# Apply our chosen conservative threshold (0.35)
decisions_conservative = ['REJECT ⚠️' if p >= 0.35 else 'APPROVE ✓' for p in probs]

# Apply default threshold (0.5) for comparison
decisions_default = ['REJECT ⚠️' if p >= 0.50 else 'APPROVE ✓' for p in probs]

print("=== Loan Decisions — Chennai Branch ===")
print(f"{'Applicant':<10} {'Default Prob':>14} {'Decision (0.5)':>16} {'Decision (0.35)':>17}")
print("-" * 62)
names = ['Ravi', 'Priya', 'Meena']
for name, prob, d05, d035 in zip(names, probs, decisions_default, decisions_conservative):
    print(f"{name:<10} {prob*100:>13.1f}% {d05:>16} {d035:>17}")

print("\nNote: Meena is borderline — at 0.35 she gets flagged for manual review.")
print("A loan officer would look at her full file before making the final call.")

## Final Summary

### What we built
A KNN pipeline that predicts loan default risk for a Chennai bank.

### What we learned

| Decision | What we did | Why |
|---|---|---|
| Scaling | StandardScaler in Pipeline | KNN uses distance — scaling is mandatory |
| Pipeline | Combined scaler + KNN | Prevents data leakage |
| Best K | Cross-validated F1 | More reliable than single test score |
| Threshold | Lowered to 0.35 | Missing defaulters costs more than false alarms |
| Evaluation | CV + classification report + AUC | Full picture of model performance |

### The business insight
In credit risk, **recall matters more than precision**.  
The threshold is a business decision — not just a math one.  
Borderline applicants (between 0.35–0.50) should go to manual review.

### KNN limitations seen here
- Slow on large datasets (computes distance to all training points at prediction)
- Sensitive to irrelevant features — feature selection helps
- No feature importance — we can't tell the bank *why* a person was rejected